In [ ]:
# =========================================================
# Imports et configuration
# =========================================================
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from config import RANDOM_STATE, TEST_SIZE, TARGET_REG, FEATURES_REG
from data import load_data, clean_data

sns.set_theme(style="whitegrid")

# =========================================================
# 1. Chargement, Analyse des Proportions et Stratégie
# =========================================================
df = load_data()
df_clean = clean_data(df)
total = len(df_clean)

print("=== ANALYSE STATISTIQUE DES VALEURS ABERRANTES (OUTLIERS) ===")
print(f"Taille du jeu de données brut : {total} morceaux\n")

cond_dur = (df_clean['duration_min'] < 2.3) | (df_clean['duration_min'] > 5.0)
cond_loud = df_clean['loudness'] < -20.0
cond_tempo = df_clean['tempo'] == 0
cond_speech = df_clean['speechiness'] >= 0.5

analyse_outliers = pd.DataFrame({
    'Critère aberrant': ['Durée < 2.3m ou > 5.0m', 'Loudness < -20 dB', 'Tempo = 0 BPM', 'Speechiness >= 0.5'],
    'Nombre de pistes': [cond_dur.sum(), cond_loud.sum(), cond_tempo.sum(), cond_speech.sum()],
    'Proportion (%)': [round(cond_dur.mean()*100, 2), round(cond_loud.mean()*100, 2), round(cond_tempo.mean()*100, 2), round(cond_speech.mean()*100, 2)],
    'Stratégie': ['Suppression', 'Suppression', 'Suppression', 'Suppression']
})

display(analyse_outliers)

print("\n=== DÉBAT : SUPPRESSION VS REMPLACEMENT ===")
print("Conclusion : Ces anomalies ou choix stricts (format radio 2.3-5 min) représentant une proportion maîtrisée du volume global, la suppression définitive est la méthode la plus rigoureuse.\n")

# Création du dataset purifié de base
df_base = df_clean[
    (df_clean['duration_min'] >= 2.3) & (df_clean['duration_min'] <= 5.0) &
    (df_clean['loudness'] >= -20.0) & (df_clean['tempo'] > 0) & (df_clean['speechiness'] < 0.5)
].copy()

def get_metrics(name, model, X_te, y_te):
    pred = model.predict(X_te)
    return {
        'Modèle': name,
        'R²': round(r2_score(y_te, pred), 3),
        'MAE': round(mean_absolute_error(y_te, pred), 3),
        'RMSE': round(np.sqrt(mean_squared_error(y_te, pred)), 3)
    }

# =========================================================
# PARTIE A : PRÉDICTION DE L'ÉNERGIE
# =========================================================
print("\n" + "="*50)
print(" PARTIE A : PRÉDICTION DE L'ÉNERGIE")
print("="*50)

# Splits Énergie
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(df_base[['loudness']], df_base[TARGET_REG], test_size=TEST_SIZE, random_state=RANDOM_STATE)
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(df_base[['acousticness']], df_base[TARGET_REG], test_size=TEST_SIZE, random_state=RANDOM_STATE)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(df_base[FEATURES_REG], df_base[TARGET_REG], test_size=TEST_SIZE, random_state=RANDOM_STATE)

# Modèles Linéaires Énergie
model_lr_loud = LinearRegression().fit(X_train_l, y_train_l)
model_lr_aco = LinearRegression().fit(X_train_a, y_train_a)
model_lr_multi = LinearRegression().fit(X_train_m, y_train_m)

# Modèles Random Forest Énergie
model_rf_loud = RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=100).fit(X_train_l, y_train_l)
model_rf_aco = RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=100).fit(X_train_a, y_train_a)
model_rf_multi = RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=100).fit(X_train_m, y_train_m)

results_nrj = pd.DataFrame([
    get_metrics("Linéaire Simple (Loudness)", model_lr_loud, X_test_l, y_test_l),
    get_metrics("Linéaire Simple (Acousticness)", model_lr_aco, X_test_a, y_test_a),
    get_metrics("Linéaire Multiple", model_lr_multi, X_test_m, y_test_m),
    get_metrics("RF Simple (Loudness)", model_rf_loud, X_test_l, y_test_l),
    get_metrics("RF Simple (Acousticness)", model_rf_aco, X_test_a, y_test_a),
    get_metrics("RF Multiple", model_rf_multi, X_test_m, y_test_m)
])
display(results_nrj)

# =========================================================
# PARTIE B : PRÉDICTION DE LA POPULARITÉ (BONUS)
# =========================================================
print("\n" + "="*50)
print(" PARTIE B : PRÉDICTION DE LA POPULARITÉ (BONUS)")
print("="*50)

TARGET_POP = 'track_popularity'

# Splits Popularité (strictement les mêmes variables : FEATURES_REG)
X_train_pop_l, X_test_pop_l, y_train_pop_l, y_test_pop_l = train_test_split(df_base[['loudness']], df_base[TARGET_POP], test_size=TEST_SIZE, random_state=RANDOM_STATE)
X_train_pop_d, X_test_pop_d, y_train_pop_d, y_test_pop_d = train_test_split(df_base[['danceability']], df_base[TARGET_POP], test_size=TEST_SIZE, random_state=RANDOM_STATE)
X_train_pop_m, X_test_pop_m, y_train_pop_m, y_test_pop_m = train_test_split(df_base[FEATURES_REG], df_base[TARGET_POP], test_size=TEST_SIZE, random_state=RANDOM_STATE)

# Entraînement Popularité
model_pop_lr_loud = LinearRegression().fit(X_train_pop_l, y_train_pop_l)
model_pop_lr_dance = LinearRegression().fit(X_train_pop_d, y_train_pop_d)
model_pop_lr_multi = LinearRegression().fit(X_train_pop_m, y_train_pop_m)
model_pop_rf_multi = RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=100, max_depth=15).fit(X_train_pop_m, y_train_pop_m)

results_pop = pd.DataFrame([
    get_metrics("Linéaire Simple (Loudness)", model_pop_lr_loud, X_test_pop_l, y_test_pop_l),
    get_metrics("Linéaire Simple (Danceability)", model_pop_lr_dance, X_test_pop_d, y_test_pop_d),
    get_metrics("Linéaire Multiple (Audio complet)", model_pop_lr_multi, X_test_pop_m, y_test_pop_m),
    get_metrics("RF Multiple (Audio complet)", model_pop_rf_multi, X_test_pop_m, y_test_pop_m)
])

display(results_pop)

print("\n=== RÉPONSE À LA QUESTION BONUS ===")
print("La précision s'effondre. La popularité ne dépend pas que des attributs audio, mais d'éléments externes (marketing, notoriété, tendances TikTok, etc.) que la machine ne possède pas.")

# =========================================================
# 5. Sauvegarde de tous les modèles finaux
# =========================================================
os.makedirs('../models', exist_ok=True)
joblib.dump(model_lr_multi, '../models/regression.joblib')
joblib.dump(model_rf_multi, '../models/rf_regression.joblib')
joblib.dump(model_pop_lr_multi, '../models/pop_lr_regression.joblib')
joblib.dump(model_pop_rf_multi, '../models/pop_rf_regression.joblib')
print("\nTous les modèles (Énergie et Popularité) ont été sauvegardés avec succès.")

Taille initiale : 28356 morceaux
Taille après Filtre A (Loudness) : 28273 morceaux
Taille après Filtre B (Acousticness) : 28193 morceaux
Taille après Filtre Global (Multiple) : 28135 morceaux

--- Performances des modèles sur leurs jeux de Test respectifs ---
                        Modèle     R²    MAE   RMSE
0            Simple (Loudness)  0.455  0.107  0.134
1        Simple (Acousticness)  0.308  0.126  0.155
2  Multiple (Toutes variables)  0.632  0.087  0.110

Modèle multiple sauvegardé dans '../models/regression.joblib'
